In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình màu sắc cho biểu đồ nhìn 'pro' hơn
sns.set_theme(style="whitegrid")
print("Đã load đồ nghề thành công!")

In [ ]:
# Đọc file CSV (Nhớ sửa lại tên file và đường dẫn cho đúng máy em nhé)
file_path = '../data/paysim_dummy.csv' 
df = pd.read_csv(file_path)

# In ra 5 dòng đầu tiên để xem hình thù dữ liệu
display(df.head())

In [ ]:
# Đếm số lượng giao dịch Bình thường (0) và Gian lận (1)
fraud_counts = df['isFraud'].value_counts()
print(fraud_counts)

# Tính tỉ lệ phần trăm
fraud_percentage = (fraud_counts[1] / len(df)) * 100
print(f"Tỉ lệ giao dịch gian lận: {fraud_percentage:.3f}%")

# Vẽ biểu đồ cột
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='isFraud', palette=['#10b981', '#ef4444']) # Xanh cho an toàn, Đỏ cho gian lận
plt.title('Sự mất cân bằng giữa Giao dịch Bình thường và Gian lận')
plt.xlabel('isFraud (0: An toàn, 1: Gian lận)')
plt.ylabel('Số lượng giao dịch')
plt.show()

In [ ]:
# Tạo bản sao để không làm hỏng dữ liệu gốc
df_processed = df.copy()

# 1. Tính toán sai lệch số dư của người gửi
# Bình thường: oldbalanceOrg - amount = newbalanceOrig
# Suy ra sai lệch = newbalanceOrig + amount - oldbalanceOrg
df_processed['errorBalanceOrig'] = df_processed['newbalanceOrig'] + df_processed['amount'] - df_processed['oldbalanceOrg']

# 2. Tính toán sai lệch số dư của người nhận
df_processed['errorBalanceDest'] = df_processed['oldbalanceDest'] + df_processed['amount'] - df_processed['newbalanceDest']

print("Đã tạo xong 2 đặc trưng 'tố giác' lừa đảo: errorBalanceOrig và errorBalanceDest")
display(df_processed[['amount', 'oldbalanceOrg', 'newbalanceOrig', 'errorBalanceOrig', 'isFraud']].head(10))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# 1. Phân chia Features (X) và Target (y)
X = df_processed.drop(['isFraud', 'isFlaggedFraud', 'nameOrig', 'nameDest'], axis=1) # Bỏ các cột không cần thiết
y = df_processed['isFraud']

# 2. Định nghĩa các cột số và cột chữ
numeric_features = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest']
categorical_features = ['type']

# 3. Tạo bộ tiền xử lý (Preprocessor)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 4. Gom mọi thứ vào Pipeline (Kèm thuật toán Random Forest)
# Thêm class_weight='balanced' để báo cho AI biết dữ liệu đang bị mất cân bằng
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

# 5. Chia tập Train/Test (80% học, 20% thi)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Kích thước tập Train:", X_train.shape)
print("Kích thước tập Test:", X_test.shape)
print("Đã sẵn sàng để huấn luyện!")

In [ ]:
# Cho cỗ máy học dữ liệu
print("Đang huấn luyện Rừng ngẫu nhiên... Vui lòng chờ...")
rf_pipeline.fit(X_train, y_train)
print("Huấn luyện thành công! Mô hình đã sẵn sàng dự đoán.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Bắt mô hình dự đoán trên tập thi (Test set)
y_pred = rf_pipeline.predict(X_test)

# 2. In bảng điểm chi tiết
print("=== BÁO CÁO ĐÁNH GIÁ MÔ HÌNH ===")
print(classification_report(y_test, y_pred, target_names=['An Toàn (0)', 'Gian Lận (1)']))

# 3. Vẽ Ma trận nhầm lẫn (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['An Toàn', 'Gian Lận'])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap='Blues', values_format='d', ax=ax)
plt.title('Ma trận Nhầm lẫn (Confusion Matrix)')
plt.grid(False) # Tắt lưới mặc định của seaborn để số hiển thị rõ hơn
plt.show()

In [ ]:
import numpy as np
import seaborn as sns

# 1. Trích xuất mô hình Random Forest từ trong lòng Pipeline
rf_model = rf_pipeline.named_steps['classifier']

# 2. Lấy tên các đặc trưng sau khi đã đi qua bộ tiền xử lý (như One-Hot)
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()

# 3. Lấy điểm số "đóng góp" của từng đặc trưng
importances = rf_model.feature_importances_

# 4. Sắp xếp từ cao xuống thấp
indices = np.argsort(importances)[::-1]

# 5. Vẽ biểu đồ
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=feature_names[indices], palette='magma')
plt.title('Mức độ ảnh hưởng của các Đặc trưng (Feature Importances)')
plt.xlabel('Điểm quan trọng (Từ 0 đến 1)')
plt.ylabel('Tên Đặc trưng (Feature)')
plt.tight_layout()
plt.show()

In [ ]:
import joblib
import os

# 1. Tạo thư mục models nếu chưa có
os.makedirs('../models', exist_ok=True)

# 2. Lưu toàn bộ Pipeline (bao gồm cả Preprocessor và Random Forest)
model_path = '../models/fraud_detection_pipeline.pkl'
joblib.dump(rf_pipeline, model_path)

print(f"✅ Đã lưu Pipeline thành công tại: {model_path}")
print("📦 File này đã bao gồm: StandardScaler, OneHotEncoder và RandomForestClassifier.")

In [ ]:
import pandas as pd
import joblib

# 1. Backend load cỗ máy phân tích (Chỉ load 1 lần khi khởi động server)
loaded_pipeline = joblib.load('../models/fraud_detection_pipeline.pkl')

# 2. Giả lập nhận cục JSON từ form React của Hiền gửi lên
# Một giao dịch nghi ngờ: Rút cạn 500 triệu
incoming_request = {
    "step": 1,
    "type": "TRANSFER",
    "amount": 500000.0,
    "oldbalanceOrg": 500000.0,
    "newbalanceOrig": 0.0,
    "oldbalanceDest": 10000.0,
    "newbalanceDest": 510000.0
}

# 3. Backend tính toán thêm các Feature Engineering (Bắt buộc)
error_balance_orig = incoming_request["newbalanceOrig"] + incoming_request["amount"] - incoming_request["oldbalanceOrg"]
error_balance_dest = incoming_request["oldbalanceDest"] + incoming_request["amount"] - incoming_request["newbalanceDest"]

# 4. Đóng gói thành DataFrame 1 dòng để chuẩn bị đưa vào Model
df_inference = pd.DataFrame([{
    "step": incoming_request["step"],
    "type": incoming_request["type"],
    "amount": incoming_request["amount"],
    "oldbalanceOrg": incoming_request["oldbalanceOrg"],
    "newbalanceOrig": incoming_request["newbalanceOrig"],
    "oldbalanceDest": incoming_request["oldbalanceDest"],
    "newbalanceDest": incoming_request["newbalanceDest"],
    "errorBalanceOrig": error_balance_orig,
    "errorBalanceDest": error_balance_dest
}])

# 5. Model ra quyết định
# Hàm predict_proba trả về xác suất [Xác_suất_An_toàn, Xác_suất_Gian_lận]
probabilities = loaded_pipeline.predict_proba(df_inference)[0]
fraud_prob = probabilities[1] # Lấy tỉ lệ phần trăm gian lận

print("=== KẾT QUẢ TỪ MODEL ===")
print(f"Xác suất gian lận: {fraud_prob * 100:.2f}%")

if fraud_prob > 0.8:
    print("Risk Level: HIGH - CHẶN GIAO DỊCH LẬP TỨC!")
elif fraud_prob > 0.4:
    print("Risk Level: MEDIUM - CẦN REVIEW THỦ CÔNG")
else:
    print("Risk Level: LOW - GIAO DỊCH HỢP LỆ")